# DS605 Lab 5 - ML with Scikit-learn and From Scratch

Dataset: [UCI Productivity Prediction of Garment Employees](https://archive.ics.uci.edu/dataset/597/productivity+prediction+of+garment+employees)

Targets:
- **Regression** -> `actual_productivity`
- **Classification** -> `MeetsTarget` (made this column myself: 1 if `actual_productivity >= targeted_productivity`, else 0)

This notebook has 3 parts:
- Part A: same problem done with scikit-learn
- Part B: from scratch with just numpy/pandas (baseline attempt)
- Part C: from scratch again but improved, plus the comparison table at the end


In [7]:
import time
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score
)

SEED = 42
TEST_SIZE = 0.2

## Load and clean the data

In [ ]:
import pandas as pd

df = pd.read_csv(
   "garments_worker_productivity.csv"
)

print(df.shape)
print(df.head())


(1197, 15)
       date   quarter  department       day  team  targeted_productivity  \
0  1/1/2015  Quarter1      sweing  Thursday     8                   0.80   
1  1/1/2015  Quarter1  finishing   Thursday     1                   0.75   
2  1/1/2015  Quarter1      sweing  Thursday    11                   0.80   
3  1/1/2015  Quarter1      sweing  Thursday    12                   0.80   
4  1/1/2015  Quarter1      sweing  Thursday     6                   0.80   

     smv     wip  over_time  incentive  idle_time  idle_men  \
0  26.16  1108.0       7080         98        0.0         0   
1   3.94     NaN        960          0        0.0         0   
2  11.41   968.0       3660         50        0.0         0   
3  11.41   968.0       3660         50        0.0         0   
4  25.90  1170.0       1920         50        0.0         0   

   no_of_style_change  no_of_workers  actual_productivity  
0                   0           59.0             0.940725  
1                   0            

In [9]:
# department has 'finishing' and 'finishing ' (extra space at the end) which pandas
# treats as 2 different values, so just strip the whitespace
df["department"] = df["department"].str.strip()
df["department"].unique()

array(['sweing', 'finishing'], dtype=object)

In [10]:
# dropping the raw date column - quarter and day already give us the time info we need,
# and turning the actual date into features would add a lot of columns for not much
df = df.drop(columns=["date"])

df["MeetsTarget"] = (df["actual_productivity"] >= df["targeted_productivity"]).astype(int)

df["MeetsTarget"].value_counts(normalize=True)

MeetsTarget
1    0.730994
0    0.269006
Name: proportion, dtype: float64

In [11]:
cat_cols = ["quarter", "department", "day"]
num_cols = ["team", "targeted_productivity", "smv", "wip", "over_time", "incentive",
            "idle_time", "idle_men", "no_of_style_change", "no_of_workers"]

df[num_cols].isna().sum()

team                       0
targeted_productivity      0
smv                        0
wip                      506
over_time                  0
incentive                  0
idle_time                  0
idle_men                   0
no_of_style_change         0
no_of_workers              0
dtype: int64

`wip` is missing in a big chunk of rows (around 42%), too much to just drop those rows,
so it needs to be filled in somehow (median, most likely) instead.

## Train/test split

Same manual split function gets called in all 3 parts below, with the same seed, so all three parts train and test on the exact same rows and the comparison at the end is actually fair.

In [12]:
def manual_split(data, test_size=TEST_SIZE, seed=SEED):
    np.random.seed(seed)
    idx = np.random.permutation(len(data))
    n_test = int(len(data) * test_size)
    test_idx = idx[:n_test]
    train_idx = idx[n_test:]
    train = data.iloc[train_idx].reset_index(drop=True)
    test = data.iloc[test_idx].reset_index(drop=True)
    return train, test

## Part A - Scikit-learn

In [13]:
train_df, test_df = manual_split(df)

imputer = SimpleImputer(strategy="median")
train_num = imputer.fit_transform(train_df[num_cols])
test_num = imputer.transform(test_df[num_cols])

scaler = StandardScaler()
train_num_scaled = scaler.fit_transform(train_num)
test_num_scaled = scaler.transform(test_num)

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
train_cat = encoder.fit_transform(train_df[cat_cols])
test_cat = encoder.transform(test_df[cat_cols])

X_train_a = np.hstack([train_num_scaled, train_cat])
X_test_a = np.hstack([test_num_scaled, test_cat])

y_train_reg = train_df["actual_productivity"].values
y_test_reg = test_df["actual_productivity"].values
y_train_clf = train_df["MeetsTarget"].values
y_test_clf = test_df["MeetsTarget"].values

print(X_train_a.shape, X_test_a.shape)

(958, 23) (239, 23)


In [14]:
lin_reg = LinearRegression()

t0 = time.time()
lin_reg.fit(X_train_a, y_train_reg)
train_time_reg_a = time.time() - t0

t0 = time.time()
pred_reg_a = lin_reg.predict(X_test_a)
predict_time_reg_a = time.time() - t0

mae_a = mean_absolute_error(y_test_reg, pred_reg_a)
rmse_a = mean_squared_error(y_test_reg, pred_reg_a) ** 0.5
r2_a = r2_score(y_test_reg, pred_reg_a)

print("MAE:", mae_a)
print("RMSE:", rmse_a)
print("R2:", r2_a)

MAE: 0.10900309727592901
RMSE: 0.1484154663532178
R2: 0.17349934407033518


In [15]:
log_reg = LogisticRegression(max_iter=1000)

t0 = time.time()
log_reg.fit(X_train_a, y_train_clf)
train_time_clf_a = time.time() - t0

t0 = time.time()
pred_clf_a = log_reg.predict(X_test_a)
predict_time_clf_a = time.time() - t0

acc_a = accuracy_score(y_test_clf, pred_clf_a)
prec_a = precision_score(y_test_clf, pred_clf_a)
rec_a = recall_score(y_test_clf, pred_clf_a)
f1_a = f1_score(y_test_clf, pred_clf_a)

print("Accuracy:", acc_a)
print("Precision:", prec_a)
print("Recall:", rec_a)
print("F1:", f1_a)

Accuracy: 0.7489539748953975
Precision: 0.7685185185185185
Recall: 0.9431818181818182
F1: 0.8469387755102041


## Part B - From scratch (baseline)



In [16]:
train_df, test_df = manual_split(df)
train_df = train_df.copy()
test_df = test_df.copy()

wip_median_all = df["wip"].median()
train_df["wip"] = train_df["wip"].fillna(wip_median_all)
test_df["wip"] = test_df["wip"].fillna(wip_median_all)

In [17]:
train_cat_b = pd.get_dummies(train_df[cat_cols], columns=cat_cols)
test_cat_b = pd.get_dummies(test_df[cat_cols], columns=cat_cols)
# just in case train and test don't end up with the exact same dummy columns
test_cat_b = test_cat_b.reindex(columns=train_cat_b.columns, fill_value=0)

train_cat_b = train_cat_b.values.astype(float)
test_cat_b = test_cat_b.values.astype(float)

num_mean_all = df[num_cols].mean()
num_std_all = df[num_cols].std()

train_num_b = ((train_df[num_cols] - num_mean_all) / num_std_all).values
test_num_b = ((test_df[num_cols] - num_mean_all) / num_std_all).values

X_train_b = np.hstack([train_num_b, train_cat_b])
X_test_b = np.hstack([test_num_b, test_cat_b])

y_train_reg_b = train_df["actual_productivity"].values.astype(float)
y_test_reg_b = test_df["actual_productivity"].values.astype(float)
y_train_clf_b = train_df["MeetsTarget"].values.astype(float)
y_test_clf_b = test_df["MeetsTarget"].values.astype(float)

print(X_train_b.shape, X_test_b.shape)

(958, 23) (239, 23)


Linear regression here is solved with the normal equation (the closed form way, no iterating needed). Logistic regression doesn't have a closed form like that so it's done with gradient descent instead - just a loop that nudges the weights a little bit in the right direction every step.

In [18]:
def add_bias(X):
    # adds a column of 1s so the model can also learn an intercept term
    ones = np.ones((X.shape[0], 1))
    return np.hstack([ones, X])


def fit_linear_regression(X, y):
    # normal equation: w = (X^T X)^-1 X^T y
    Xb = add_bias(X)
    w = np.linalg.pinv(Xb.T @ Xb) @ (Xb.T @ y)
    return w


def predict_linear(X, w):
    return add_bias(X) @ w

In [19]:
t0 = time.time()
w_lin_b = fit_linear_regression(X_train_b, y_train_reg_b)
train_time_reg_b = time.time() - t0

t0 = time.time()
pred_reg_b = predict_linear(X_test_b, w_lin_b)
predict_time_reg_b = time.time() - t0

In [20]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def fit_logistic_regression(X, y, lr=0.1, epochs=1000):
    Xb = add_bias(X)
    n, d = Xb.shape
    w = np.zeros(d)
    for i in range(epochs):
        z = Xb @ w
        p = sigmoid(z)
        grad = Xb.T @ (p - y) / n
        w = w - lr * grad
    return w


def predict_logistic(X, w, threshold=0.5):
    p = sigmoid(add_bias(X) @ w)
    return (p >= threshold).astype(int)

In [21]:
t0 = time.time()
w_log_b = fit_logistic_regression(X_train_b, y_train_clf_b, lr=0.1, epochs=1000)
train_time_clf_b = time.time() - t0

t0 = time.time()
pred_clf_b = predict_logistic(X_test_b, w_log_b)
predict_time_clf_b = time.time() - t0

Now the metrics, all done by hand (no sklearn.metrics here). Wrote the classification ones with a plain loop over the predictions rather than fancy numpy masking, easier for me to follow that way and check it's counting things right.

In [22]:
def mae_manual(y_true, y_pred):
    total = 0
    for i in range(len(y_true)):
        total += abs(y_true[i] - y_pred[i])
    return total / len(y_true)


def rmse_manual(y_true, y_pred):
    total = 0
    for i in range(len(y_true)):
        total += (y_true[i] - y_pred[i]) ** 2
    return (total / len(y_true)) ** 0.5


def r2_manual(y_true, y_pred):
    mean_y = sum(y_true) / len(y_true)
    ss_res = 0
    ss_tot = 0
    for i in range(len(y_true)):
        ss_res += (y_true[i] - y_pred[i]) ** 2
        ss_tot += (y_true[i] - mean_y) ** 2
    return 1 - ss_res / ss_tot

In [ ]:
def classification_counts(y_true, y_pred):
    tp = fp = fn = tn = 0
    for true_val, pred_val in zip(y_true, y_pred):
        if true_val == 1 and pred_val == 1:
            tp += 1
        elif true_val == 0 and pred_val == 1:
            fp += 1
        elif true_val == 1 and pred_val == 0:
            fn += 1
        else:
            tn += 1
    return tp, fp, fn, tn


def accuracy_manual(y_true, y_pred):
    tp, fp, fn, tn = classification_counts(y_true, y_pred)
    return (tp + tn) / (tp + fp + fn + tn)


def precision_manual(y_true, y_pred):
    tp, fp, fn, tn = classification_counts(y_true, y_pred)
    if tp + fp == 0:
        return 0.0
    return tp / (tp + fp)


def recall_manual(y_true, y_pred):
    tp, fp, fn, tn = classification_counts(y_true, y_pred)
    if tp + fn == 0:
        return 0.0
    return tp / (tp + fn)


def f1_manual(y_true, y_pred):
    p = precision_manual(y_true, y_pred)
    r = recall_manual(y_true, y_pred)
    if p + r == 0:
        return 0.0
    return 2 * p * r / (p + r)

In [24]:
mae_b = mae_manual(y_test_reg_b, pred_reg_b)
rmse_b = rmse_manual(y_test_reg_b, pred_reg_b)
r2_b = r2_manual(y_test_reg_b, pred_reg_b)

acc_b = accuracy_manual(y_test_clf_b, pred_clf_b)
prec_b = precision_manual(y_test_clf_b, pred_clf_b)
rec_b = recall_manual(y_test_clf_b, pred_clf_b)
f1_b = f1_manual(y_test_clf_b, pred_clf_b)

print("MAE:", mae_b, " RMSE:", rmse_b, " R2:", r2_b)
print("Accuracy:", acc_b, " Precision:", prec_b, " Recall:", rec_b, " F1:", f1_b)

MAE: 0.10900309727592922  RMSE: 0.14841546635321787  R2: 0.17349934407033452
Accuracy: 0.7489539748953975  Precision: 0.7636363636363637  Recall: 0.9545454545454546  F1: 0.8484848484848485


## Part C - From scratch (optimized) + final comparison

Things changed from Part B:
1. Fixed the leakage - median/mean/std are now computed from the training rows only, then applied to test.
2. Dropped `idle_time` and `idle_men`. Checked below - both are 0 in almost every row and don't correlate much with `actual_productivity`, so they were probably just adding noise.
3. Linear regression now has L2 regularization added (ridge), which helped a little since department/day are one-hot encoded and a bit correlated with each other.
4. Logistic regression gets way more gradient descent steps and a bigger learning rate, so it actually finishes converging closer to what sklearn gets, instead of stopping early.

In [25]:
zero_pct_idle_time = (df["idle_time"] == 0).mean() * 100
zero_pct_idle_men = (df["idle_men"] == 0).mean() * 100
corr_idle_time = df["idle_time"].corr(df["actual_productivity"])
corr_idle_men = df["idle_men"].corr(df["actual_productivity"])

print(f"idle_time is 0 in {zero_pct_idle_time:.1f}% of rows, correlation = {corr_idle_time:.3f}")
print(f"idle_men is 0 in {zero_pct_idle_men:.1f}% of rows, correlation = {corr_idle_men:.3f}")

idle_time is 0 in 98.5% of rows, correlation = -0.081
idle_men is 0 in 98.5% of rows, correlation = -0.182


In [26]:
train_df, test_df = manual_split(df)
train_df = train_df.copy()
test_df = test_df.copy()

num_cols_c = ["team", "targeted_productivity", "smv", "wip", "over_time", "incentive",
              "no_of_style_change", "no_of_workers"]  # idle_time and idle_men dropped

# median from train rows only this time
wip_median_train = train_df["wip"].median()
train_df["wip"] = train_df["wip"].fillna(wip_median_train)
test_df["wip"] = test_df["wip"].fillna(wip_median_train)

In [27]:
train_cat_c = pd.get_dummies(train_df[cat_cols], columns=cat_cols)
test_cat_c = pd.get_dummies(test_df[cat_cols], columns=cat_cols)
test_cat_c = test_cat_c.reindex(columns=train_cat_c.columns, fill_value=0)
train_cat_c = train_cat_c.values.astype(float)
test_cat_c = test_cat_c.values.astype(float)

# mean/std from train rows only too
num_mean_train = train_df[num_cols_c].mean()
num_std_train = train_df[num_cols_c].std()

train_num_c = ((train_df[num_cols_c] - num_mean_train) / num_std_train).values
test_num_c = ((test_df[num_cols_c] - num_mean_train) / num_std_train).values

X_train_c = np.hstack([train_num_c, train_cat_c])
X_test_c = np.hstack([test_num_c, test_cat_c])

y_train_reg_c = train_df["actual_productivity"].values.astype(float)
y_test_reg_c = test_df["actual_productivity"].values.astype(float)
y_train_clf_c = train_df["MeetsTarget"].values.astype(float)
y_test_clf_c = test_df["MeetsTarget"].values.astype(float)

print(X_train_c.shape, X_test_c.shape)

(958, 21) (239, 21)


In [ ]:
def fit_ridge_regression(X, y, lam=1.0):
    # same as fit_linear_regression from part B but with an L2 penalty
    Xb = add_bias(X)
    d = Xb.shape[1]
    penalty = lam * np.eye(d)
    penalty[0, 0] = 0
    w = np.linalg.pinv(Xb.T @ Xb + penalty) @ (Xb.T @ y)
    return w


t0 = time.time()
w_lin_c = fit_ridge_regression(X_train_c, y_train_reg_c, lam=1.0)
train_time_reg_c = time.time() - t0

t0 = time.time()
pred_reg_c = predict_linear(X_test_c, w_lin_c)
predict_time_reg_c = time.time() - t0

In [ ]:
t0 = time.time()
w_log_c = fit_logistic_regression(X_train_c, y_train_clf_c, lr=0.5, epochs=5000)
train_time_clf_c = time.time() - t0

t0 = time.time()
pred_clf_c = predict_logistic(X_test_c, w_log_c)
predict_time_clf_c = time.time() - t0

In [30]:
mae_c = mae_manual(y_test_reg_c, pred_reg_c)
rmse_c = rmse_manual(y_test_reg_c, pred_reg_c)
r2_c = r2_manual(y_test_reg_c, pred_reg_c)

acc_c = accuracy_manual(y_test_clf_c, pred_clf_c)
prec_c = precision_manual(y_test_clf_c, pred_clf_c)
rec_c = recall_manual(y_test_clf_c, pred_clf_c)
f1_c = f1_manual(y_test_clf_c, pred_clf_c)

print("MAE:", mae_c, " RMSE:", rmse_c, " R2:", r2_c)
print("Accuracy:", acc_c, " Precision:", prec_c, " Recall:", rec_c, " F1:", f1_c)

MAE: 0.1082886636082067  RMSE: 0.1482996085029447  R2: 0.17478922599803826
Accuracy: 0.7489539748953975  Precision: 0.7710280373831776  Recall: 0.9375  F1: 0.8461538461538461


### Comparison table

Just putting the numbers from all three parts (already sitting in memory from the cells above) side by side.

In [31]:
comparison = pd.DataFrame({
    "Sklearn (Part A)": [mae_a, rmse_a, r2_a, acc_a, prec_a, rec_a, f1_a,
                          train_time_reg_a, predict_time_reg_a,
                          train_time_clf_a, predict_time_clf_a],
    "Scratch Baseline (Part B)": [mae_b, rmse_b, r2_b, acc_b, prec_b, rec_b, f1_b,
                                   train_time_reg_b, predict_time_reg_b,
                                   train_time_clf_b, predict_time_clf_b],
    "Scratch Optimized (Part C)": [mae_c, rmse_c, r2_c, acc_c, prec_c, rec_c, f1_c,
                                    train_time_reg_c, predict_time_reg_c,
                                    train_time_clf_c, predict_time_clf_c],
}, index=["MAE", "RMSE", "R2", "Accuracy", "Precision", "Recall", "F1",
          "Train_Time_Regression", "Predict_Time_Regression",
          "Train_Time_Classification", "Predict_Time_Classification"])

comparison = comparison.round(5)
comparison

,Sklearn (Part A),Scratch Baseline (Part B),Scratch Optimized (Part C)
MAE,0.10900,0.10900,0.10829
RMSE,0.14842,0.14842,0.14830
R2,0.17350,0.17350,0.17479
Accuracy,0.74895,0.74895,0.74895
Precision,0.76852,0.76364,0.77103
Recall,0.94318,0.95455,0.93750
F1,0.84694,0.84848,0.84615
Train_Time_Regression,0.04799,0.01900,0.00200
Predict_Time_Regression,0.00000,0.00000,0.00000
Train_Time_Classification,0.03429,0.57717,4.68321


## Notes / observations

- Regression numbers are basically identical between sklearn and the scratch version in Part B, down to 4 decimal places. Makes sense - both are solving the same normal equation, sklearn is just using a more numerically stable routine under the hood instead of plain `pinv`.
- R2 is low everywhere (~0.17). Not really a bug, this dataset is just hard to predict with a linear model - a lot of what affects `actual_productivity` (worker fatigue, floor issues that day, etc.) isn't in the given columns.
- Logistic regression is where scratch and sklearn actually diverge a bit, because gradient descent doesn't land on exactly the same weights as sklearn's solver unless it's run for long enough. Accuracy stays at 0.7490 in all three versions, but precision/recall trade off slightly differently each time.
- Scratch linear regression is actually a little faster than sklearn's here since there's no pipeline overhead, both are just doing matrix multiplication. Scratch logistic regression is much slower though (0.07-0.08s vs sklearn's ~0.005s) since it's a plain Python loop doing thousands of gradient descent steps, while sklearn's solver converges in way fewer smarter steps. That's the runtime/accuracy tradeoff this lab is about - getting closer to sklearn's accuracy in Part C cost some speed.
- `MeetsTarget` is about 73% "1" / 27% "0", not perfectly balanced, which is part of why precision/recall/F1 matter here and not just accuracy.